<img src="https://raw.githubusercontent.com/MohammedAly22/qwencleo-asr/main/assets/QwenCleo-ASR-Banner.png" width="100%"/>

# 🎙️ QwenCleo-ASR — vLLM Serving & True Streaming

Egyptian Arabic & code-switching speech recognition, built on Qwen3-ASR.

[Model](https://huggingface.co/mohammedaly22/QwenCleo-ASR) · [GitHub](https://github.com/MohammedAly22/qwencleo-asr) · [PyPI](https://pypi.org/project/qwencleo-asr/)

> **Runtime → Change runtime type → GPU** before running. Then run the cells top to bottom.

QwenCleo inherits Qwen3-ASR's **token-by-token streaming** via vLLM.

> ⚠️ **CUDA requirement.** vLLM's Qwen3-ASR support ships in the **nightly** wheel, which is built against **CUDA 13** (`libcudart.so.13`). It will only run on a runtime that provides CUDA 13. Standard Colab GPU runtimes ship CUDA 12.x, where the vLLM nightly **fails to import** (`ImportError: libcudart.so.13`).

> ✅ **If your runtime is CUDA 12.x** (most Colab GPUs today), use the [FastAPI server](https://colab.research.google.com/github/MohammedAly22/qwencleo-asr/blob/main/examples/03_fastapi_server.ipynb) or [Quick Start](https://colab.research.google.com/github/MohammedAly22/qwencleo-asr/blob/main/examples/01_quickstart.ipynb) notebook instead — they don't need vLLM and give the same transcriptions. Run this notebook on a **CUDA-13 GPU** (e.g. a newer cloud box) for true streaming.

## 0. Check your CUDA version first

If this shows CUDA **12.x**, the vLLM nightly below will not import — switch to the FastAPI notebook.

In [ ]:
!nvcc --version 2>/dev/null | grep release || nvidia-smi | grep -i cuda

## 1. Install vLLM (nightly) + the client

`--index-strategy` is a `uv` flag (not pip), so we install `uv` first.

In [ ]:
# vLLM nightly has day-0 Qwen3-ASR support (needs CUDA 13)
!pip install -q uv
!uv pip install --system -q -U vllm --pre \
    --extra-index-url https://wheels.vllm.ai/nightly \
    --index-strategy unsafe-best-match
!uv pip install --system -q 'vllm[audio]' openai httpx
!pip install -q qwencleo-asr --no-deps

# fail fast with a clear message if the runtime lacks CUDA 13
import importlib, sys
try:
    importlib.import_module('vllm')
    print('✅ vLLM imported — runtime CUDA is compatible')
except ImportError as e:
    if 'libcudart.so.13' in str(e):
        sys.exit('❌ This runtime is CUDA 12.x; vLLM nightly needs CUDA 13. Use the FastAPI / Quick Start notebook instead.')
    raise

## 2. Download the model first

Pull weights now so the server boots fast in the next cell.

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download('mohammedaly22/QwenCleo-ASR')
print('model downloaded')

## 3. Start a vLLM server in the background

Logs go to `vllm.log` so we can show the reason if startup fails. The memory flags keep it within a single (small) Colab GPU. First start can take a few minutes while vLLM compiles + loads the model.

In [ ]:
import subprocess, time, requests

logf = open('vllm.log', 'w')
proc = subprocess.Popen(
    ['vllm', 'serve', 'mohammedaly22/QwenCleo-ASR',
     '--gpu-memory-utilization', '0.9',
     '--max-model-len', '4096',
     '--dtype', 'bfloat16'],
    stdout=logf, stderr=subprocess.STDOUT)
print('starting vLLM server (logging to vllm.log)...')

up = False
for _ in range(120):          # up to ~10 min
    try:
        requests.get('http://localhost:8000/v1/models', timeout=2)
        up = True; print('✅ server up'); break
    except Exception:
        if proc.poll() is not None:
            print('❌ vLLM exited — last 40 log lines:\n')
            print(''.join(open('vllm.log').readlines()[-40:]))
            raise RuntimeError('vLLM failed to start (see log above)')
        time.sleep(5)
if not up:
    print(''.join(open('vllm.log').readlines()[-40:]))
    raise TimeoutError('vLLM did not come up in time (see log above)')

> **Debug tip:** if it failed, run `!tail -n 60 vllm.log` in a new cell. Common Colab causes: GPU too small / out of memory, or the runtime's CUDA being incompatible with the vLLM nightly. The free **T4** may be too small — try an **L4/A100** runtime, or use the FastAPI server notebook instead (no vLLM needed).

## 4. Sample audio

In [ ]:
# Grab a sample Egyptian/code-switching clip to transcribe
import urllib.request, os
URL = 'https://huggingface.co/mohammedaly22/QwenCleo-ASR/resolve/main/assets/sample.wav'
FALLBACK = 'https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-ASR-Repo/asr_en.wav'
path = 'sample.wav'
try:
    urllib.request.urlretrieve(URL, path)
except Exception:
    urllib.request.urlretrieve(FALLBACK, path)
print('saved', path, os.path.getsize(path), 'bytes')

## 5. TRUE streaming — deltas arrive token by token

In [ ]:
from qwencleo_asr import stream_vllm

for delta in stream_vllm('sample.wav', language='Arabic'):
    print(delta, end='', flush=True)
print()

## 6. One-shot via the OpenAI-compatible API

In [ ]:
from qwencleo_asr import transcribe_vllm
print(transcribe_vllm('sample.wav'))

## 7. OpenAI SDK directly (the documented form)

In [ ]:
from openai import OpenAI
client = OpenAI(base_url='http://localhost:8000/v1', api_key='EMPTY')
with open('sample.wav','rb') as f:
    print(client.audio.transcriptions.create(
        model='mohammedaly22/QwenCleo-ASR', file=f.read()).text)

## 8. Stop the server

In [ ]:
proc.terminate()